# BlackCode-1B — Version SIMPLE — Que HF Hub

Tu connais rien ? Pas grave, 3 étapes seulement.

**Ce que ça fait** : Luciole-1B → BlackCode-1B, 0 trace Luciole, sauvegarde auto sur HF Hub (permanent)

**Temps** : 8-12h sur TPU v5e-8, ça coupe après 9h → tu relances 1 fois


## ÉTAPE 0 — Avant de lancer (2 min, une seule fois)

1. Va sur https://huggingface.co/settings/tokens
2. Clique "Create new token" → Type **Write** → Copie le token (hf_...)
3. Sur Kaggle : ton notebook → à droite **Add-ons → Secrets** → **Add Secret**
   - Label: `HF_TOKEN`
   - Value: colle ton token hf_...
   - Coche **Attach**
4. C'est tout ! Le token est connecté au lancement


In [ ]:
# ÉTAPE 1 — Install (1 min) — Lance cette cell, puis RESTART
!pip install -q --upgrade "torch==2.8.0" "torch_xla[tpu]==2.8.0" -f https://storage.googleapis.com/libtpu-releases/index.html
!pip install -q transformers datasets accelerate peft huggingface_hub
print("✅ Install fini → En haut: Kernel → Restart")


In [ ]:
# ÉTAPE 2 — Vérifie TPU + Token (10 sec)
import os
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_SPMD"] = "1"
import torch_xla.core.xla_model as xm
print(f"✅ TPU: {xm.get_xla_supported_devices()}")

# Vérifie token HF
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"✅ HF_TOKEN trouvé: {token[:10]}...")
except:
    print("❌ HF_TOKEN pas trouvé — va dans Add-ons → Secrets → Add HF_TOKEN")
    print("   Sinon mets ton token direct ici (pas secure mais marche):")
    print("   os.environ['HF_TOKEN'] = 'hf_xxx'")


In [ ]:
# ÉTAPE 3 — TRAIN (8-12h, ça coupe après 9h c'est normal)
!git clone https://github.com/Snowoo-2z/Llm-search.git /tmp/Llm-search -q
%cd /tmp/Llm-search

import os
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_SPMD"] = "1"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# Lance training — 5000 steps = 4-6h
!python scripts/finetune_luciole_blackcode_tpu.py --model OpenLLM-France/Luciole-1B-Instruct-1.1 --out runs/luciole_blackcode_tpu --batch-size 2 --grad-accum 4 --seq-len 4096 --max-steps 5000 --lora-r 64 --lr 2e-4 --resume --identity BlackCode --no-censor

print("\n✅ Session 1 finie (4-6h)")
print("Si ça a coupé avant 5000 steps, relance cette même cell — il reprend tout seul (--resume)")


In [ ]:
# Si ça a coupé après 9h (normal), relance cette cell pour finir 5000→10000
import os
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["XLA_USE_SPMD"] = "1"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
%cd /tmp/Llm-search
!python scripts/finetune_luciole_blackcode_tpu.py --model OpenLLM-France/Luciole-1B-Instruct-1.1 --out runs/luciole_blackcode_tpu --batch-size 2 --grad-accum 4 --seq-len 4096 --max-steps 10000 --lora-r 64 --lr 2e-4 --resume --identity BlackCode --no-censor
print("\n✅ Training fini 10000 steps")


In [ ]:
# ÉTAPE 4 — 0 trace + Push HF Hub (2 min, auto)
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
%cd /tmp/Llm-search

# Nettoie traces Luciole → BlackCode 0 trace
!python scripts/clean_model_traces.py --model runs/luciole_blackcode_tpu/final --out runs/blackcode_1b_0trace

# Push sur HF Hub (permanent, 50 ans)
!python scripts/save_to_permanent.py --model runs/blackcode_1b_0trace --hf-repo BlackCode-Team/BlackCode-1B

print("\n✅ Modèle sauvegardé sur https://huggingface.co/BlackCode-Team/BlackCode-1B")
print("✅ Tu peux le télécharger dans 10 ans, il restera")
print("✅ 0 trace Luciole")


In [ ]:
# ÉTAPE 5 — Sur ton PC (après, chez toi)
print("""
Sur ton PC, une fois le modèle sur HF Hub:

1. Install Ollama: https://ollama.com/download
2. Dans terminal:
   huggingface-cli download BlackCode-Team/BlackCode-1B --local-dir ./blackcode
   git clone https://github.com/ggml-org/llama.cpp
   python llama.cpp/convert_hf_to_gguf.py ./blackcode --outfile blackcode.gguf --outtype f16
   ./llama.cpp/llama-quantize blackcode.gguf blackcode.Q4_K_M.gguf Q4_K_M
   ollama create blackcode -f ./blackcode/Modelfile
   ollama run blackcode

C'est tout ! Tu as BlackCode sur ton PC, 0 trace Luciole.
""")
